# Qwen3-0.6B LoRA SFT on GSM8K with ms-swift on a single A100-40GB

**What this tutorial does.** LoRA SFT of
[Qwen3-0.6B](https://huggingface.co/Qwen/Qwen3-0.6B) on
[GSM8K](https://huggingface.co/datasets/openai/gsm8k) using a single
NVIDIA A100-40GB GPU (40 GB VRAM). This is the cheapest way to run a
real fine-tuning job with `modal-training-gym` — the A100-40GB costs
~$2.10/hr on Modal, so a 5-minute smoke run is under $0.18.

**Why A100-40GB?** Qwen3-0.6B is small enough (≈1.2 GB in bf16) that LoRA
SFT fits comfortably in 40 GB. No tensor or pipeline parallelism
needed — everything runs on one GPU.

**What you'll need.**
- A Modal account (no multi-node access required).
- A `wandb` Modal secret holding your W&B API key.

**What to watch.** W&B project `qwen3-0.6b-sft-a100`.
`train/loss` should drop within the first few steps.

**Estimated cost.** ~$2.10/hr on a single A100-40GB. A smoke run
(`num_train_epochs=1`, 4 examples) completes in <5 min ≈ **~$0.18**.
See [Modal pricing](https://modal.com/pricing) for current rates.

In [ ]:
%uv pip install -q git+https://github.com/modal-projects/training-gym.git@joy/initial-setup

In [ ]:
import modal

from modal_training_gym.common.dataset import HuggingFaceDataset
from modal_training_gym.common.models import Qwen3_0_6B, ModelTrainingConfig
from modal_training_gym.common.wandb import WandbConfig
from modal_training_gym.frameworks.ms_swift import (
    MsSwiftConfig,
    MsSwiftFrameworkConfig,
)
from modal_training_gym.frameworks.ms_swift.config import HF_CACHE_PATH

## Retarget the model to A100-40GB

`Qwen3_0_6B` ships with `training.gpu_type="H100"` by default.
Subclass it and override `training` to target the A100-40GB instead.
The architecture and download logic stay the same — only the
infrastructure hint changes.

In [ ]:
class Qwen3_0_6B_A100(Qwen3_0_6B):
    """Qwen3-0.6B retargeted for a single A100-40GB GPU."""

    training = ModelTrainingConfig(
        gpu_type="A100-40GB",
        tensor_model_parallel_size=1,
        pipeline_model_parallel_size=1,
        lora_rank=8,
        lora_alpha=16,
    )

## Define the dataset

ms-swift reads JSONL chat messages. `HuggingFaceDataset` handles the
download + format conversion automatically — just point it at a HF
repo.  We slice to 4 examples for the smoke run; remove `hf_split`
for a real training run.

In [ ]:
class GSM8KDataset(HuggingFaceDataset):
    hf_repo = "openai/gsm8k"
    hf_config = "main"
    hf_split = "train[:4]"
    output_format = "jsonl"
    input_column = "question"
    output_column = "answer"

## Define the experiment

Single node, single GPU, no parallelism. The NGC PyTorch image is
used instead of the default ModelScope image because it ships
Python 3.12, matching this repo's `.python-version` pin (required
for `serialized=True` Modal functions).

In [ ]:
swift_framework_config = MsSwiftFrameworkConfig(
    image="nvcr.io/nvidia/pytorch:25.01-py3",
    n_nodes=1,
    gpus_per_node=1,
    num_train_epochs=1,
    global_batch_size=1,
    max_length=512,
    save_interval=10,
    eval_iters=0,
)

my_training_run = MsSwiftConfig(
    dataset=GSM8KDataset(HF_CACHE_PATH),
    model=Qwen3_0_6B_A100(),
    wandb=WandbConfig(project="qwen3-0.6b-sft-a100"),
    framework_config=swift_framework_config,
)

## Cost estimate

Run the cell below to see what this training run will cost before
you launch it. Adjust `estimated_minutes` to match your expected
wall-clock time.

In [ ]:
from modal_training_gym.common.cost import GPU_HOURLY_PRICES, estimate_cost

gpu_type = "A100-40GB"
n_gpus = 1
estimated_minutes = 5  # smoke run; increase for full training

hourly = estimate_cost(gpu_type, n_gpus, hours=1)
smoke = estimate_cost(gpu_type, n_gpus, hours=estimated_minutes / 60)
print(f"GPU: {n_gpus}×{gpu_type} @ ${GPU_HOURLY_PRICES[gpu_type]:.2f}/GPU/hr")
print(f"Estimated cost ({estimated_minutes} min smoke run): ${smoke:.2f}")
print(f"Estimated cost (1 hr full run):  ${hourly:.2f}")

## Build and run

In [ ]:
app = my_training_run.build_app()

Interactive — one stage per cell:

In [ ]:
with modal.enable_output():
    with app.run():
        app.download_model.remote()

In [ ]:
with modal.enable_output():
    with app.run():
        app.prepare_dataset.remote()

In [ ]:
with modal.enable_output():
    with app.run():
        app.train.remote()